# 05 — Capability Budgets

**Chapter 2 teaching lab — §2.5 + Figure 2.4**

> **Teaching outcome:** Budget cumulative operational consequence.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
from harnesslab.budget import ALLOWED_IN, COST, Budget, Mode, run_plan
from harnesslab.viz import budget_bar
from harnesslab.playground import budget_playground
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(5)

## Concept
Per-action checks miss **accumulation**: fifty individually harmless steps can add up to a harmful run.
A **capability budget** assigns each operation a consequence cost and caps the total per run. A warning
threshold gives early signal; the hard limit stops consequential work.

When the budget runs out, the run should **degrade**, not crash: keep doing safe work (read, draft),
stop doing consequential work, and surface the situation to a human.

## Why this matters
Retry loops, fan-out, and "one more step" are how cheap runs become expensive incidents. A budget is the
control that notices the *shape* of a run rather than any single call.

## Diagram (Figure 2.4)
```mermaid
flowchart LR
  E[EXECUTE] -->|"spent ≥ warning"| W[EXECUTE + alert]
  W -->|hard limit hit| D[DRAFT_ONLY]
  D -->|drafting unaffordable| R[READ_ONLY]
  R -->|stop condition| P[PAUSED]
  classDef ok fill:#16a34a22,stroke:#16a34a
  classDef warn fill:#d9770622,stroke:#d97706
  classDef bad fill:#dc262622,stroke:#dc2626
  class E ok
  class W,D warn
  class R,P bad
```

## Read the implementation

In [ ]:
show_source(Budget, run_plan)

## Experiment

In [ ]:
budget = Budget(limit=12, warning=8)
plan = ["search", "search", "draft", "external_message", "production_write", "draft", "external_message", "search"]
mode, trace = run_plan(plan, budget)
budget_bar(trace, COST, budget.limit, budget.warning)

## Expected behavior

In [ ]:
assert budget.spent <= budget.limit
assert mode is Mode.DRAFT_ONLY
executed = [op for op, status, _ in trace if status in ("ok", "warning")]
assert "production_write" not in executed
after = trace[[op for op, _, _ in trace].index("production_write") + 1:]
assert all(op in ALLOWED_IN[Mode.DRAFT_ONLY] or status == "refused" for op, status, _ in after)
print(f"Budget held at {budget.spent}/{budget.limit}; the run degraded to {mode.value} instead of failing.")

## Red-team round (§2.5)
The same attacks run against two defense sets: **HarnessLab** (what this lab built) and
**Shortcuts** (the patterns the chapter warns about). Every attack must be blocked by HarnessLab.
The Shortcuts defense checks each call against the limit but never adds them up.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE, "2.5"), run_gauntlet(NAIVE, "2.5")
assert all(r.blocked for r in secure), [r.attack.name for r in secure if not r.blocked]
gauntlet_board(secure, shortcuts, title="Gauntlet · §2.5")

## Playground
Click operations to build a run. Watch the warning line, the exhaustion, and the refusals.

In [ ]:
budget_playground()

## Challenge
Add a separate **retry budget** for one tool (say, 3 retries of `external_message`) alongside the run
budget. Show a trace where the run budget has room but the retry budget stops the loop. Then add the
`READ_ONLY` transition: when should a run degrade past `DRAFT_ONLY`?